# MIMIC-IV static pipeline for time-to-discharge
This notebook performs subject-level split, train-only preprocessing, and exports X/y train-val-test files.


In [13]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ==============================
# 0) Configuration
# ==============================
BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "MIMIC-IV-static(Group Assignment).csv"
OUTPUT_DIR = BASE_DIR / "mimic_static_split_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Current working directory:", BASE_DIR)
print("Input exists:", INPUT_PATH.exists())
print("Output dir:", OUTPUT_DIR)


# Target definition
TARGET_COL = "time_to_discharge_hours"
LOG_TARGET_COL = "log_time_to_discharge_hours"
EVENT_COL = "icu_death_flag"
KEEP_ONLY_LIVE_DISCHARGE = True

# Split configuration
RANDOM_STATE = 42
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15
assert abs(TRAIN_SIZE + VAL_SIZE + TEST_SIZE - 1.0) < 1e-8

ID_COLS = ["subject_id", "hadm_id", "stay_id"]
TARGET_COLS = [TARGET_COL, LOG_TARGET_COL]

# Columns used to derive the target or that directly leak the outcome
LEAKAGE_COLS = [
    "intime",
    "outtime",
    "deathtime",
    "los",
    "icu_los_hours",
    "hospital_expire_flag",
    "icu_death_flag",
]

HIGH_MISSING_THRESHOLD = 0.5
MISSING_INDICATOR_LOWER = 0.2
MISSING_INDICATOR_UPPER = 0.5
DATETIME_PARSE_THRESHOLD = 0.8


# ==============================
# 1) Basic helpers
# ==============================
def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.str.strip().str.lower()
        .str.replace(r"[^\w]+", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )
    return df


def inspect_data(df: pd.DataFrame) -> None:
    print("=" * 90)
    print("1) Basic data inspection")
    print("=" * 90)
    print(f"Shape: {df.shape}")
    print("\nData types:")
    print(df.dtypes.value_counts())

    missing_rate = (df.isna().mean() * 100).sort_values(ascending=False)
    print("\nTop 20 columns by missing rate (%):")
    print(missing_rate.head(20).round(2))


def drop_duplicate_rows(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates().copy()
    after = len(df)

    print("\n2) Remove duplicate rows")
    print(f"Rows before: {before}")
    print(f"Rows after : {after}")
    print(f"Duplicates removed: {before - after}")
    return df


# ==============================
# 2) Target construction
# ==============================
def build_time_to_discharge_target(
    df: pd.DataFrame,
    target_col: str,
    log_target_col: str,
    event_col: str = "icu_death_flag",
    keep_only_live_discharge: bool = True,
) -> pd.DataFrame:
    df = df.copy()
    print("\n3) Build regression target: ICU time-to-discharge")

    available_cols = set(df.columns)

    if "icu_los_hours" in available_cols:
        df[target_col] = pd.to_numeric(df["icu_los_hours"], errors="coerce")
        print("Target source: icu_los_hours")
    elif "los" in available_cols:
        los_numeric = pd.to_numeric(df["los"], errors="coerce")
        median_los = los_numeric.dropna().median()
        if pd.notna(median_los) and median_los <= 30:
            df[target_col] = los_numeric * 24
            print("Target source: los (assumed days, converted to hours)")
        else:
            df[target_col] = los_numeric
            print("Target source: los (assumed already in hours)")
    elif {"intime", "outtime"}.issubset(available_cols):
        intime = pd.to_datetime(df["intime"], errors="coerce")
        outtime = pd.to_datetime(df["outtime"], errors="coerce")
        df[target_col] = (outtime - intime).dt.total_seconds() / 3600
        print("Target source: outtime - intime (hours)")
    else:
        raise ValueError(
            "Cannot construct target. Need one of: icu_los_hours, los, or intime+outtime."
        )

    invalid_mask = df[target_col].isna() | (df[target_col] <= 0)
    print(f"Rows removed due to missing/non-positive target: {int(invalid_mask.sum())}")
    df = df.loc[~invalid_mask].copy()

    if keep_only_live_discharge:
        if event_col not in df.columns:
            raise ValueError(
                f"'{event_col}' not found. Cannot restrict to live ICU discharge."
            )
        before = len(df)
        df = df[df[event_col] == 0].copy()
        print(f"Rows kept with live ICU discharge ({event_col} == 0): {len(df)} / {before}")
    else:
        print("All ICU stays retained, including deaths.")

    df[log_target_col] = np.log1p(df[target_col])

    print("\nTarget summary (hours):")
    print(df[target_col].describe().round(2))
    print("\nTarget summary (log1p hours):")
    print(df[log_target_col].describe().round(4))
    return df


def drop_leakage_columns(df: pd.DataFrame, target_col: str, leakage_cols: list[str]) -> pd.DataFrame:
    print("\n4) Drop leakage columns")
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found.")

    leakage_to_drop = [c for c in leakage_cols if c in df.columns and c != target_col]
    print(f"Leakage columns removed: {leakage_to_drop}")
    return df.drop(columns=leakage_to_drop, errors="ignore").copy()


# ==============================
# 3) Race regrouping
# ==============================
def map_race_to_major_group(value) -> str:
    if pd.isna(value):
        return "Unknown"

    s = str(value).strip().upper()

    if s in {"", "UNKNOWN", "UNABLE TO OBTAIN", "PATIENT DECLINED TO ANSWER", "?"}:
        return "Unknown"

    if "HISPANIC" in s or "LATINO" in s or "SOUTH AMERICAN" in s:
        return "Hispanic"

    if "BLACK" in s:
        return "Black"

    if "ASIAN" in s:
        return "Asian"

    if (
        "AMERICAN INDIAN" in s
        or "ALASKA NATIVE" in s
        or "NATIVE HAWAIIAN" in s
        or "PACIFIC ISLANDER" in s
    ):
        return "Native"

    if "WHITE" in s or s == "PORTUGUESE":
        return "White"

    if "MULTIPLE" in s or s == "OTHER":
        return "Other"

    return "Other"


def regroup_race(df: pd.DataFrame, race_col: str = "race") -> pd.DataFrame:
    df = df.copy()
    print("\n5) Regroup race categories")

    if race_col not in df.columns:
        print(f"Column '{race_col}' not found. Skip regrouping.")
        return df

    print("Original race categories (top 20):")
    print(df[race_col].value_counts(dropna=False).head(20))

    df[race_col] = df[race_col].apply(map_race_to_major_group)

    print("\nRegrouped race categories:")
    print(df[race_col].value_counts(dropna=False))
    return df


# ==============================
# 4) Subject-level split
# ==============================
def split_by_subject(
    df: pd.DataFrame,
    subject_col: str = "subject_id",
    train_size: float = 0.70,
    val_size: float = 0.15,
    test_size: float = 0.15,
    random_state: int = 42,
):
    print("\n6) Subject-level train / val / test split")
    if subject_col not in df.columns:
        raise ValueError(f"'{subject_col}' not found in dataframe.")

    unique_subjects = df[subject_col].dropna().unique()
    train_subjects, temp_subjects = train_test_split(
        unique_subjects,
        test_size=(1 - train_size),
        random_state=random_state,
        shuffle=True,
    )

    relative_test_size = test_size / (val_size + test_size)
    val_subjects, test_subjects = train_test_split(
        temp_subjects,
        test_size=relative_test_size,
        random_state=random_state,
        shuffle=True,
    )

    train_subjects = set(train_subjects)
    val_subjects = set(val_subjects)
    test_subjects = set(test_subjects)

    train_df = df[df[subject_col].isin(train_subjects)].copy()
    val_df = df[df[subject_col].isin(val_subjects)].copy()
    test_df = df[df[subject_col].isin(test_subjects)].copy()

    overlap_train_val = train_subjects.intersection(val_subjects)
    overlap_train_test = train_subjects.intersection(test_subjects)
    overlap_val_test = val_subjects.intersection(test_subjects)
    assert len(overlap_train_val) == 0
    assert len(overlap_train_test) == 0
    assert len(overlap_val_test) == 0

    print(f"Unique subjects - train: {len(train_subjects)}, val: {len(val_subjects)}, test: {len(test_subjects)}")
    print(f"Rows - train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

    return train_df, val_df, test_df


# ==============================
# 5) Train-only preprocessing helpers
# ==============================
def fit_datetime_columns(
    train_df: pd.DataFrame,
    protected_cols: list[str],
    parse_threshold: float = DATETIME_PARSE_THRESHOLD,
) -> list[str]:
    name_hints = ("time", "date", "chart")
    candidate_cols = train_df.select_dtypes(include=["object"]).columns.tolist()
    candidate_cols = [
        c for c in candidate_cols
        if c not in protected_cols and any(hint in c for hint in name_hints)
    ]

    datetime_cols = []
    for col in candidate_cols:
        parsed = pd.to_datetime(train_df[col], errors="coerce")
        parse_rate = parsed.notna().mean()
        if parse_rate >= parse_threshold:
            datetime_cols.append(col)
    return datetime_cols


def transform_datetime_columns(df: pd.DataFrame, datetime_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in datetime_cols:
        if col not in df.columns:
            continue
        parsed = pd.to_datetime(df[col], errors="coerce")
        df[f"{col}_hour"] = parsed.dt.hour
        df[f"{col}_weekday"] = parsed.dt.weekday
        df[f"{col}_month"] = parsed.dt.month
        df = df.drop(columns=[col])
    return df


def split_variable_types(df: pd.DataFrame, target_cols: list[str], id_cols: list[str]):
    feature_cols = [c for c in df.columns if c not in id_cols + target_cols]
    categorical_cols = df[feature_cols].select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]
    return numeric_cols, categorical_cols


def clean_unreasonable_values(df: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    rules = {
        "age": (0, 120),
        "heart_rate": (20, 250),
        "sbp": (40, 300),
        "dbp": (20, 200),
        "mbp": (20, 220),
        "resp_rate": (3, 80),
        "temperature": (30, 43),
        "spo2": (0, 100),
        "glucose": (10, 1000),
        "gcs": (3, 15),
        "hematocrit": (0, 80),
        "hemoglobin": (0, 30),
        "platelets": (0, 2000),
        "wbc": (0, 500),
        "albumin": (0, 10),
        "globulin": (0, 12),
        "total_protein": (0, 15),
        "aniongap": (-50, 100),
        "bicarbonate": (0, 60),
        "bun": (0, 300),
        "calcium": (0, 20),
        "chloride": (50, 200),
        "creatinine": (0, 30),
        "sodium": (100, 200),
        "potassium": (1, 10),
        "bilirubin": (0, 100),
        "inr": (0, 20),
        "ptt": (0, 200),
        "pt": (0, 200),
        "po2": (0, 600),
        "so2": (0, 100),
    }

    total_changed = 0
    for col in numeric_cols:
        if col not in df.columns:
            continue

        lower, upper = None, None
        for key, value_range in rules.items():
            if key in col:
                lower, upper = value_range
                break

        if lower is not None:
            invalid_mask = (df[col].notna()) & ((df[col] < lower) | (df[col] > upper))
            changed = int(invalid_mask.sum())
            if changed > 0:
                df.loc[invalid_mask, col] = np.nan
                total_changed += changed

    print(f"Unreasonable values converted to NaN: {total_changed}")
    return df


def fit_missingness_rules(
    train_df: pd.DataFrame,
    protected_cols: list[str],
    high_missing_threshold: float,
    indicator_lower: float,
    indicator_upper: float,
):
    missing_ratio = train_df.isnull().mean().sort_values(ascending=False)

    drop_cols = missing_ratio[missing_ratio > high_missing_threshold].index.tolist()
    drop_cols = [c for c in drop_cols if c not in protected_cols]

    indicator_cols = []
    for col, miss_rate in missing_ratio.items():
        if col in protected_cols or col in drop_cols:
            continue
        if indicator_lower < miss_rate <= indicator_upper:
            indicator_cols.append(col)

    return {
        "drop_cols": drop_cols,
        "indicator_cols": indicator_cols,
        "missing_ratio": missing_ratio.to_dict(),
    }


def apply_missingness_rules(df: pd.DataFrame, rules: dict) -> pd.DataFrame:
    df = df.copy()
    drop_cols = [c for c in rules["drop_cols"] if c in df.columns]
    df = df.drop(columns=drop_cols, errors="ignore")

    for col in rules["indicator_cols"]:
        if col in df.columns:
            df[f"{col}_missing"] = df[col].isnull().astype(int)
        else:
            df[f"{col}_missing"] = 1
    return df


def fit_iqr_bounds(df: pd.DataFrame, numeric_cols: list[str]) -> dict:
    bounds = {}
    for col in numeric_cols:
        if col not in df.columns:
            continue
        series = df[col]
        if series.notna().sum() < 5:
            continue
        unique_vals = pd.Series(series.dropna().unique())
        if len(unique_vals) <= 2 or col.endswith("_missing"):
            continue

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            continue
        bounds[col] = {
            "lower": float(q1 - 1.5 * iqr),
            "upper": float(q3 + 1.5 * iqr),
        }
    return bounds


def apply_iqr_clipping(df: pd.DataFrame, bounds: dict) -> pd.DataFrame:
    df = df.copy()
    for col, lims in bounds.items():
        if col in df.columns:
            df[col] = df[col].clip(lower=lims["lower"], upper=lims["upper"])
    return df


def fit_imputers(df: pd.DataFrame, numeric_cols: list[str], categorical_cols: list[str]):
    numeric_fill = {}
    for col in numeric_cols:
        if col in df.columns:
            median_value = df[col].median()
            if pd.isna(median_value):
                median_value = 0
            numeric_fill[col] = float(median_value)

    categorical_fill = {}
    for col in categorical_cols:
        if col in df.columns:
            mode_series = df[col].mode(dropna=True)
            fill_value = mode_series.iloc[0] if len(mode_series) > 0 else "Unknown"
            categorical_fill[col] = str(fill_value)

    return numeric_fill, categorical_fill


def apply_imputers(
    df: pd.DataFrame,
    numeric_fill: dict,
    categorical_fill: dict,
    numeric_cols: list[str],
    categorical_cols: list[str],
) -> pd.DataFrame:
    df = df.copy()

    for col in numeric_cols:
        if col in df.columns:
            fill_value = numeric_fill.get(col, 0)
            df[col] = df[col].fillna(fill_value)

    for col in categorical_cols:
        if col in df.columns:
            fill_value = categorical_fill.get(col, "Unknown")
            df[col] = df[col].fillna(fill_value)

    return df


def fit_category_levels(df: pd.DataFrame, categorical_cols: list[str]) -> dict:
    category_levels = {}
    for col in categorical_cols:
        if col in df.columns:
            levels = sorted(pd.Series(df[col].astype(str).dropna().unique()).tolist())
            category_levels[col] = levels
    return category_levels


def apply_one_hot_encoding(df: pd.DataFrame, category_levels: dict) -> pd.DataFrame:
    df = df.copy()
    categorical_cols = list(category_levels.keys())

    for col, levels in category_levels.items():
        if col in df.columns:
            series = df[col].astype(str)
        else:
            series = pd.Series([np.nan] * len(df), index=df.index, dtype="object")
        df[col] = pd.Categorical(series, categories=levels)

    encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False, dtype=int)
    return encoded


def align_feature_columns(X: pd.DataFrame, feature_columns: list[str]) -> pd.DataFrame:
    X = X.copy()
    for col in feature_columns:
        if col not in X.columns:
            X[col] = 0
    extra_cols = [c for c in X.columns if c not in feature_columns]
    if extra_cols:
        X = X.drop(columns=extra_cols)
    return X[feature_columns]


def fit_scaler_columns(X_train: pd.DataFrame) -> list[str]:
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scale_cols = []
    for col in numeric_cols:
        unique_vals = pd.Series(X_train[col].dropna().unique())
        if len(unique_vals) <= 2:
            continue
        scale_cols.append(col)
    return scale_cols


# ==============================
# 6) Main pipeline
# ==============================
def main():
    df = pd.read_csv(INPUT_PATH)
    df = standardize_column_names(df)
    inspect_data(df)

    df = drop_duplicate_rows(df)
    df = build_time_to_discharge_target(
        df=df,
        target_col=TARGET_COL,
        log_target_col=LOG_TARGET_COL,
        event_col=EVENT_COL,
        keep_only_live_discharge=KEEP_ONLY_LIVE_DISCHARGE,
    )
    df = drop_leakage_columns(df, TARGET_COL, LEAKAGE_COLS)
    df = regroup_race(df, race_col="race")

    # Split first to avoid patient leakage and to ensure train-only preprocessing fit.
    train_df, val_df, test_df = split_by_subject(
        df,
        subject_col="subject_id",
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )

    protected_cols = ID_COLS + TARGET_COLS

    # Datetime feature rules learned from train only.
    datetime_cols = fit_datetime_columns(train_df, protected_cols=protected_cols)
    print(f"\nDatetime columns learned from train: {datetime_cols}")
    train_df = transform_datetime_columns(train_df, datetime_cols)
    val_df = transform_datetime_columns(val_df, datetime_cols)
    test_df = transform_datetime_columns(test_df, datetime_cols)

    # Determine types from train after datetime transform.
    numeric_cols_train, categorical_cols_train = split_variable_types(
        train_df, target_cols=TARGET_COLS, id_cols=ID_COLS
    )
    print("\n7) Variable types from train (before missing handling)")
    print(f"Numeric columns     : {len(numeric_cols_train)}")
    print(f"Categorical columns : {len(categorical_cols_train)}")
    print(f"Categorical list    : {categorical_cols_train}")

    # Clean unreasonable values using the train-derived feature list.
    print("\n8) Handle unreasonable values")
    train_df = clean_unreasonable_values(train_df, numeric_cols_train)
    val_df = clean_unreasonable_values(val_df, numeric_cols_train)
    test_df = clean_unreasonable_values(test_df, numeric_cols_train)

    # Missingness rules learned from train only.
    print("\n9) Fit missingness rules on train")
    missing_rules = fit_missingness_rules(
        train_df,
        protected_cols=protected_cols,
        high_missing_threshold=HIGH_MISSING_THRESHOLD,
        indicator_lower=MISSING_INDICATOR_LOWER,
        indicator_upper=MISSING_INDICATOR_UPPER,
    )
    print(f"Columns dropped for >{HIGH_MISSING_THRESHOLD:.0%} missing (train only): {len(missing_rules['drop_cols'])}")
    print(missing_rules["drop_cols"])
    print(
        f"Columns receiving missing indicators ({MISSING_INDICATOR_LOWER:.0%}-{MISSING_INDICATOR_UPPER:.0%} missing in train): "
        f"{len(missing_rules['indicator_cols'])}"
    )
    print(missing_rules["indicator_cols"])

    train_df = apply_missingness_rules(train_df, missing_rules)
    val_df = apply_missingness_rules(val_df, missing_rules)
    test_df = apply_missingness_rules(test_df, missing_rules)

    # Recompute variable types after missing indicators / column dropping.
    numeric_cols_train, categorical_cols_train = split_variable_types(
        train_df, target_cols=TARGET_COLS, id_cols=ID_COLS
    )
    print("\n10) Variable types from train (after missing handling)")
    print(f"Numeric columns     : {len(numeric_cols_train)}")
    print(f"Categorical columns : {len(categorical_cols_train)}")

    # IQR clipping rules learned from train only.
    print("\n11) Fit IQR clipping bounds on train")
    iqr_bounds = fit_iqr_bounds(train_df, numeric_cols_train)
    print(f"Numeric columns with IQR clipping bounds: {len(iqr_bounds)}")

    train_df = apply_iqr_clipping(train_df, iqr_bounds)
    val_df = apply_iqr_clipping(val_df, iqr_bounds)
    test_df = apply_iqr_clipping(test_df, iqr_bounds)

    # Train-only imputers.
    print("\n12) Fit imputers on train")
    numeric_fill, categorical_fill = fit_imputers(train_df, numeric_cols_train, categorical_cols_train)
    train_df = apply_imputers(train_df, numeric_fill, categorical_fill, numeric_cols_train, categorical_cols_train)
    val_df = apply_imputers(val_df, numeric_fill, categorical_fill, numeric_cols_train, categorical_cols_train)
    test_df = apply_imputers(test_df, numeric_fill, categorical_fill, numeric_cols_train, categorical_cols_train)

    # Train-only category levels and one-hot encoding.
    print("\n13) Fit one-hot category levels on train")
    category_levels = fit_category_levels(train_df, categorical_cols_train)
    print(f"Categorical columns to encode: {list(category_levels.keys())}")

    train_encoded = apply_one_hot_encoding(train_df, category_levels)
    val_encoded = apply_one_hot_encoding(val_df, category_levels)
    test_encoded = apply_one_hot_encoding(test_df, category_levels)

    # Split into X / y and align columns based on train only.
    y_cols = ID_COLS + TARGET_COLS
    X_train = train_encoded.drop(columns=y_cols, errors="ignore").copy()
    X_val = val_encoded.drop(columns=y_cols, errors="ignore").copy()
    X_test = test_encoded.drop(columns=y_cols, errors="ignore").copy()

    feature_columns = X_train.columns.tolist()
    X_val = align_feature_columns(X_val, feature_columns)
    X_test = align_feature_columns(X_test, feature_columns)

    # Train-only scaling.
    print("\n14) Fit scaler on train")
    scale_cols = fit_scaler_columns(X_train)
    scaler = StandardScaler()
    if scale_cols:
        X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
        X_val[scale_cols] = scaler.transform(X_val[scale_cols])
        X_test[scale_cols] = scaler.transform(X_test[scale_cols])
    print(f"Scaled columns: {len(scale_cols)}")

    # Targets and cohort files.
    y_train = train_encoded[y_cols].copy()
    y_val = val_encoded[y_cols].copy()
    y_test = test_encoded[y_cols].copy()

    y_train["split"] = "train"
    y_val["split"] = "val"
    y_test["split"] = "test"

    cohort_master = pd.concat([y_train, y_val, y_test], axis=0, ignore_index=True)

    # Export outputs.

    X_train.to_csv(OUTPUT_DIR / "X_static_train.csv", index=False)
    X_val.to_csv(OUTPUT_DIR / "X_static_val.csv", index=False)
    X_test.to_csv(OUTPUT_DIR / "X_static_test.csv", index=False)

    y_train.to_csv(OUTPUT_DIR / "y_train.csv", index=False)
    y_val.to_csv(OUTPUT_DIR / "y_val.csv", index=False)
    y_test.to_csv(OUTPUT_DIR / "y_test.csv", index=False)

    cohort_master.to_csv(OUTPUT_DIR / "cohort_master.csv", index=False)



    manifest = {
    "input_path": str(INPUT_PATH),
    "output_dir": str(OUTPUT_DIR),
    "target_col": TARGET_COL,
    "log_target_col": LOG_TARGET_COL,
    "event_col": EVENT_COL,
    "keep_only_live_discharge": KEEP_ONLY_LIVE_DISCHARGE,
    "high_missing_threshold": HIGH_MISSING_THRESHOLD,
    "missing_indicator_lower": MISSING_INDICATOR_LOWER,
    "missing_indicator_upper": MISSING_INDICATOR_UPPER,
    "train_shape": list(X_train.shape),
    "val_shape": list(X_val.shape),
    "test_shape": list(X_test.shape),
    }

    with open(OUTPUT_DIR / "preprocessing_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)


    
    # manifest = {
    #     "input_path": INPUT_PATH,
    #     "output_dir": str(OUTPUT_DIR),
    #     "random_state": RANDOM_STATE,
    #     "split_ratio": {"train": TRAIN_SIZE, "val": VAL_SIZE, "test": TEST_SIZE},
    #     "target_col": TARGET_COL,
    #     "log_target_col": LOG_TARGET_COL,
    #     "keep_only_live_discharge": KEEP_ONLY_LIVE_DISCHARGE,
    #     "id_cols": ID_COLS,
    #     "datetime_cols_from_train": datetime_cols,
    #     "dropped_high_missing_cols_from_train": missing_rules["drop_cols"],
    #     "missing_indicator_source_cols_from_train": missing_rules["indicator_cols"],
    #     "categorical_cols_from_train": list(category_levels.keys()),
    #     "num_features": len(feature_columns),
    #     "feature_columns": feature_columns,
    #     "scaled_columns": scale_cols,
    #     "train_rows": int(len(X_train)),
    #     "val_rows": int(len(X_val)),
    #     "test_rows": int(len(X_test)),
    #     "train_subjects": int(y_train["subject_id"].nunique()),
    #     "val_subjects": int(y_val["subject_id"].nunique()),
    #     "test_subjects": int(y_test["subject_id"].nunique()),
    # }

    # with open(OUTPUT_DIR / "preprocessing_manifest.json", "w", encoding="utf-8") as f:
    #     json.dump(manifest, f, ensure_ascii=False, indent=2)

    print("\n" + "=" * 90)
    print("15) Export completed")
    print("=" * 90)
    print(f"X_train shape: {X_train.shape}")
    print(f"X_val shape  : {X_val.shape}")
    print(f"X_test shape : {X_test.shape}")
    print(f"Saved outputs to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()


Current working directory: /Users/tianmanyan/Desktop/SPH6004 Assignment/Assignment2_Group
Input exists: True
Output dir: /Users/tianmanyan/Desktop/SPH6004 Assignment/Assignment2_Group/mimic_static_split_outputs
1) Basic data inspection
Shape: (76943, 142)

Data types:
float64    125
object      10
int64        7
Name: count, dtype: int64

Top 20 columns by missing rate (%):
ast_min                   100.00
ast_max                   100.00
inr_max                   100.00
inr_min                   100.00
so2_max                   100.00
so2_min                   100.00
nrbc_max                   99.95
nrbc_min                   99.95
d_dimer_min                99.29
d_dimer_max                99.29
ggt_min                    99.29
ggt_max                    99.29
globulin_max               98.06
globulin_min               98.06
bilirubin_indirect_max     96.88
bilirubin_indirect_min     96.88
total_protein_max          96.86
total_protein_min          96.86
bilirubin_direct_max       96